# 03. RQ3, what happens at 96 and 200 training languages

> Reference notebook. Every number below is produced by the same functions that
> write the paper's tables (`paper_resources/scripts/`), so this is a readable
> view of the pipeline rather than a second implementation. Regenerate the
> `.tex` files with `python scripts/run_all.py`.


**Question.** Do the configurations selected on ten languages still lead once
training covers 96 and 200 languages?

Backs Table 2 (final ranking and verdicts), Table 3 (selection transfer),
Table A11 (tests against the base model), and Table A12 (omnibus tests).


In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in Path.cwd().resolve().parents if (p / "paper_resources").is_dir())
sys.path.insert(0, str(REPO_ROOT / "paper_resources" / "scripts"))

import common as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
# 4 significant digits rather than 4 decimals, so that a p-value of 5.6e-15 stays
# readable next to an effect size of 0.94.
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

cfg = C.load_config()
BENCHMARKS = cfg["analysis"]["benchmarks"]
ALPHA = cfg["analysis"]["alpha"]
print(f"repo root : {REPO_ROOT}")
print(f"benchmarks: {BENCHMARKS}")
print(f"alpha     : {ALPHA}")

repo root : /data/upb/users/n/nikit/profiles/unix/cs/projects/moe-study-analysis
benchmarks: ['Belebele', 'FLORES', 'SIB200', 'Taxi1500']
alpha     : 0.05


Positions are comparable **within** a regime and not across the two, because
the regimes use different evaluation populations, so the base model's own
numbers differ.


In [2]:
import tab5_rq3_final as rq3

baseline = cfg["rq3"]["baseline_run_id"]
regimes = cfg["rq3"]["regimes"]
panels = {r: C.load_final_panel(C.results_path(cfg, f"final_{r}_xlsx"), cfg) for r in regimes}

for regime, panel in panels.items():
    print(f"{regime:>4} training languages: {panel['language'].nunique():>3} evaluated languages, "
          f"{len(panel):>5} rows, {panel['run_id'].nunique()} systems")

  96 training languages:  80 evaluated languages,  1494 rows, 6 systems
 200 training languages: 136 evaluated languages,  2448 rows, 6 systems


In [3]:
rankings = {r: C.final_ranking(panels[r], BENCHMARKS) for r in regimes}
labels = {cfg["rq3"]["baseline_run_id"]: cfg["rq3"]["baseline_label"]}
labels.update({e["run_id"]: e["label"] for e in cfg["rq3"]["finalists"]})

for regime in regimes:
    print(f"=== {regime} training languages (lower is better)")
    display(rankings[regime].rename(index=labels).round(2))

=== 96 training languages (lower is better)


score_family,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,
MOELoRA r2/32 k8,3.48,2.56,1.44,4.13,2.9
MOELoRA r2/16 k8,3.49,3.16,3.54,2.32,3.13
MTL-LoRA r8/16,4.08,2.88,4.81,2.47,3.56
Base model,1.78,5.21,3.34,3.98,3.58
HydraLoRA r2,3.24,3.98,3.16,4.64,3.75
AdaMoE r2/8 k6,4.95,3.22,4.71,3.46,4.08


=== 200 training languages (lower is better)


score_family,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,
HydraLoRA r2,3.53,2.07,1.59,3.51,2.67
Base model,1.51,4.44,3.27,3.94,3.29
MOELoRA r2/32 k8,3.11,3.52,3.68,3.01,3.33
MTL-LoRA r8/16,3.75,3.96,3.24,3.18,3.53
AdaMoE r2/8 k6,5.08,2.59,3.64,4.27,3.9
MOELoRA r2/16 k8,4.02,4.42,5.58,3.1,4.28


## The unadapted model is hard to beat

The base model places fourth of six at 96 training languages and **second of
six** at 200. No configuration is best on more than two of the four benchmarks
in either regime, and none is best in both regimes.


In [4]:
tests = {r: C.final_pairwise_tests(panels[r], BENCHMARKS) for r in regimes}
omnibus = pd.concat(
    {r: tests[r].groupby("benchmark", sort=False).first().reset_index() for r in regimes},
    names=["regime"],
).reset_index(level=0)
omnibus[["regime", "benchmark", "n_units", "friedman_q", "kendalls_w", "p_friedman"]]

,regime,benchmark,n_units,friedman_q,kendalls_w,p_friedman
0,96,Belebele,40,63.11,0.3155,2.767e-12
1,96,FLORES,100,132.2,0.2644,8.068e-27
2,96,SIB200,51,111.8,0.4385,1.683e-22
3,96,Taxi1500,58,78.67,0.2713,1.593e-15
0,200,Belebele,65,130.4,0.4012,1.971e-26
1,200,FLORES,166,228.1,0.2748,2.793e-47
2,200,SIB200,84,197.9,0.4711,8.144e-41
3,200,Taxi1500,93,37.69,0.08105,4.357e-07


Kendall's *W* is worth reading before any ordering. At 200 languages Taxi1500's
evaluation cases barely agree on any ordering at all (*W* of about 0.08), so its
ranking is uninformative rather than a tie.


## The selection stage did not predict either outcome

This is the paper's headline. The configuration ranked first after the
ten-language follow-up finishes third in both broader regimes.


In [5]:
transfer = pd.read_csv(C.output_dir(cfg, "data") / "tab7_rq3_transfer.csv")
display(transfer)
pd.read_csv(C.output_dir(cfg, "data") / "tab7_rq3_transfer_correlations.csv")

,run_id,label,followup_rank_of_22,followup,96,200
0,mtllora_lr5e5_r8_nup16,MTL-LoRA r8/16,1,1,3,3
1,moelora_tr32_e16_k8,MOELoRA r2/16 k8,2,2,2,5
2,hydralora_r2,HydraLoRA r2,3,3,4,1
3,moelora_tr64_e32_k8,MOELoRA r2/32 k8,4,4,1,2
4,mola_tr16_e8_k6,AdaMoE r2/8 k6,5,5,5,4


,pair,spearman_rho
0,followup vs 96,0.3
1,followup vs 200,-0.1
2,96 vs 200,0


Rank-order agreement across stages is essentially absent, with rho of 0.30 from
ten languages to 96, rho of -0.10 from ten to 200, and rho of 0.00 from 96 to
200. With five configurations these are descriptive rather than inferential, but
they match what the tables show.

**The reduced-scale search identified a workable region without identifying the
ordering inside it.**
